In [39]:
import networkx as nx 
import pandas as pd 
import numpy as np
import pickle
import os
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from kneed import KneeLocator
# scipy dependency for PPR

from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.vectors import ListVector, StrVector

In [40]:
# Load all network

def pickle_load(in_file):
    with open(in_file,'rb') as input_f:
        var = pickle.load(input_f)
    return var

string_nx  = pickle_load("../data/network/string.pkl")
biogrid_nx = pickle_load("../data/network/biogrid.pkl")
bioplex_nx = pickle_load("../data/network/bioplex.pkl")
humanet_nx = pickle_load("../data/network/humannet.pkl")


In [41]:
seed_nodes = pd.read_csv("../cfg/seed_nodes_CP_sym.txt", header=None).values.flatten()
print(seed_nodes)

['PTPRD' 'STK38' 'PXT1' 'KCTD20' 'ETV7']


In [42]:
def ppr_network(seed_nodes, network, name):
    common_nodes = set(seed_nodes) & set(network.nodes)
    print(f"{len(common_nodes)} nodes in common, i.e.: {common_nodes}")

    # build personalized
    personalization = {
        seed: 1/len(common_nodes) for seed in common_nodes
    }

    assert len(set(network.nodes)) == len(list(network.nodes))

    ppr = nx.pagerank(
        G=network,
        personalization=personalization,
    )

    mean_ = np.mean(list(ppr.values()))
    std_ = np.std(list(ppr.values()))

    ppr_df = pd.DataFrame.from_dict(ppr, orient='index', columns=['ppr_importance'])
    ppr_df['zscore'] = (ppr_df['ppr_importance'] - mean_)/std_
    ppr_df = ppr_df.sort_values(['ppr_importance'], ascending=False)
    ppr_df['rank'] = list(range(1,len(ppr_df)+1))
    ppr_df = ppr_df.add_suffix('_'+name)
    return ppr_df
        

In [43]:
networks = {
    'string':   string_nx,
    'biogrid':  biogrid_nx,
    'bioplex':  bioplex_nx,
    'humannet': humanet_nx
}

ppr_results = {}

for k,v in tqdm(networks.items()):
    print(v)
    ppr_results[k] =  ppr_network(seed_nodes, v, k)


  0%|          | 0/4 [00:00<?, ?it/s]

Graph with 12421 nodes and 130974 edges
3 nodes in common, i.e.: {'ETV7', 'STK38', 'PTPRD'}
Graph with 11938 nodes and 95824 edges
4 nodes in common, i.e.: {'KCTD20', 'ETV7', 'STK38', 'PTPRD'}
Graph with 13957 nodes and 118162 edges
4 nodes in common, i.e.: {'KCTD20', 'ETV7', 'STK38', 'PTPRD'}
Graph with 18459 nodes and 977495 edges
5 nodes in common, i.e.: {'KCTD20', 'STK38', 'PTPRD', 'ETV7', 'PXT1'}


In [44]:
merged_df = pd.DataFrame()

for k,df in ppr_results.items():
    merged_df = (
        merged_df.merge(
            df,
            how='outer',
            right_index=True,
            left_index=True,
            validate='one_to_one'
        )
    )

In [45]:
# i want genes that are in at least 3 networks
# threshold controls the minimum number of networks

merged_df = merged_df[(merged_df.isna()).sum(axis=1) <= 3] # 3 as there are 3 values for each net and there are 4 networks
merged_df.head()

merged_df.to_parquet("../output/ppr_results.parquet", compression="zstd")

In [46]:
merged_df_ranks = merged_df[['rank_string','rank_biogrid','rank_bioplex','rank_humannet']]
merged_df_scores = merged_df[['ppr_importance_string','ppr_importance_biogrid','ppr_importance_bioplex','ppr_importance_humannet']]
merged_df_ranks.head()

,rank_string,rank_biogrid,rank_bioplex,rank_humannet
A1BG,NaN,11764.0,6352.0,11345.0
A1CF,10698.0,9020.0,13745.0,11714.0
A2M,8938.0,4825.0,5320.0,1473.0
A4GALT,7352.0,NaN,9205.0,12886.0
AAAS,4017.0,7039.0,8135.0,3584.0


In [47]:
r_lists = {}

for col in merged_df_ranks.columns:
    r_lists[col] = StrVector(merged_df_ranks[col].sort_values().dropna().index.tolist())

# Create R list
r_rank_list = ListVector(r_lists)

In [48]:
# Import R package
rankaggreg = importr("RobustRankAggreg")

In [49]:
# Call RankAggreg
result = rankaggreg.aggregateRanks(
    glist = r_rank_list,
    full = True,
    exact = True,
    method = 'RRA'
)
result = pandas2ri.rpy2py(result)

In [50]:
top_genes = result.iloc[:int(len(result)*0.01)]
top_genes.to_csv("../output/top_1perc_genes_RRA.csv")